In [26]:
import numpy as np
import pandas as pd
import plotly.express as px

def rename_columns(df):
    df.columns = [col.split('#')[-1].strip() if '#' in col else col.strip() for col in df.columns]
    return df

In [ ]:
df = rename_columns(pd.read_csv("1WhenSuperconduct.csv", comment = ";"))
df.groupby().mean(numeric_only=True).reset_index()

px.line(df, x="time.time",y="ThetaK", color = "repeat")

# Time

There are 3 different ways to get the time in python:
- time.time()  only formats time.
- datetime.datetime.now()  does date and time and timezone conversions
- pandas.Timestamp.now()   pandas copy of the datetime module.

Probably best to stick to the first and last Here's how to use them:

## Time Recording

In [23]:
import pandas as pd
import time
import numpy as np

In [19]:
time.time() # seconds since the last epoch. Timezone independent (technically utc)

1735414581.1708066

In [20]:
pd.Timestamp.now() # defaults to local time, but doesn't save time zone

Timestamp('2024-12-28 11:36:21.179799')

In [21]:
pd.Timestamp.now(tz='UTC') #specify time zone to record data
pd.Timestamp.now(tz='America/Los_Angeles')
pd.Timestamp.now(tz='America/New_York')

Timestamp('2024-12-28 14:36:21.186962-0500', tz='America/New_York')

## Time Reading / converting
pandas imports all 3 formats. However, if you want to plot in plotly in a human readable way, you'll want to convert.
Here's how.

In [29]:
# Fake example data
df = pd.DataFrame({
    'time.time': [time.time() + i * 3600 for i in range(10)],
    'fake_data': np.random.rand(10)})

# Convert time.time to datetime, interpret as utc, then convert to LA time
df['LAtime'] = pd.to_datetime(df['time.time'], unit='s').dt.tz_localize('UTC').dt.tz_convert('America/Los_Angeles')

In [ ]:
# works in plots like 
# px.line(df, x="LAtime",y="fake_data")

In [32]:
df.dtypes["LAtime"]

datetime64[ns, America/Los_Angeles]

## Finding separate runs from breaks in the data

In [ ]:
df['runNum'] = (df['time.time'].diff() > 600).cumsum()

## Hysteresis updown sweep separation

In [ ]:
# break up into upsweeps and down sweeps
window_size=41 # periododicity+1 to garuntee a smooth curve.
df14["sBz"] = df14["Bz"].rolling(window=window_size, center=True).mean()
df14["updown"] = (df14["sBz"].diff() >0)*2-1



import pandas as pd

def segregate_hyster(df, column, window_size):
    """
    Adds smoothed column and up/down sweep indicator to the DataFrame.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        column (str): The column name to process.
        window_size (int): The rolling window size for smoothing.

    Returns:
        pd.DataFrame: A DataFrame with added smoothed column and up/down sweep indicator.
    """
    smoothed_column = f"s{column}"
    updown_column = f"{column}_updown"

    df[smoothed_column] = df[column].rolling(window=window_size, center=True).mean()
    df[updown_column] = (df[smoothed_column].diff() > 0) * 2 - 1

    return df

# Plotly 
[Plot options](https://chatgpt.com/share/6772d903-b8e0-8004-8e86-077779e521bf)

In [ ]:
px.line(df33, x = "now", y='myHF2LI.dc',
        color = "mag.Bz",
        hover_data=["myHF2LI.dc"]
        ).show()

Plot everything

In [ ]:
for col in df.columns:
    px.scatter(df, x='Time Stamp (sec)', y = col ).show()
print(df.columns)

Heatmaps

In [ ]:
## Heatmap Examples



px.scatter(df17_avg, x= "By", y="Bz", color=toPlot).show()


px.density_heatmap( df17_avg, x="By", y="Bz", z=toPlot, histfunc="avg").show()

# px.density_contour( df17_avg, x="By", y="Bz", z=toPlot, histfunc="avg"
# ).update_traces(contours_coloring="fill").show()

hide all options

In [ ]:
fig.update_traces(visible='legendonly')
fig.for_each_trace(lambda t: t.update(visible='legendonly') if "dashed" in t.name else None)
fig.for_each_trace(lambda t: t.update(visible='legendonly') if "A" in t.name else None)

Ticks

In [ ]:
fig.update_xaxes(dtick=90, rangemode="tozero").update_yaxes(rangemode="tozero")

Combine plots

In [ ]:
px.line(df1, x="year", y="pop", title="Combined Figure") \
    .add_traces(px.line(df2, x="year", y="pop").data) 

# Fitting

In [ ]:
fig = px.line(df26_avg, x="X2", y="Y2", line_dash="myHF2LI.dc") # The data
fig.add_scatter(x=[df26_avg["X2"].min(), 0], y=np.polyval(np.polyfit(df26_avg["X2"], df26_avg["Y2"], 1), [df26_avg["X2"].min(), 0]), mode="lines", name="Best Fit") #linear fit
fig.show()

Linear background subtraction

In [ ]:

coefs = np.polyfit(test.Bz, test.ThetaK, 1)
test['ThetaK_sub'] = test.ThetaK - np.polyval(coefs, test.Bz)
px.line(test, x='Bz', y='ThetaK_sub', title="background subtracted optics", color = "repeat").show()

# Folding

In [ ]:
df["sign"] = np.where(df["I"] >= 0, "pos", "neg")  # Identify sign
df["absI"] = df["I"].abs()                        # For pivot index
pv = df.pivot_table(index="absI", columns="sign", values="V", aggfunc="mean")
pv["V_antisym"] = (pv["pos"] - pv["neg"]) / 2      # [V(+I) - V(-I)] / 2
pv["V_sym"]     = (pv["pos"] + pv["neg"]) / 2      # [V(+I) + V(-I)] / 2
pv.dropna(inplace=True)                            # Remove absI rows missing pos or neg


# Iterate over all variables in the pivoted table
for var in pv.columns.get_level_values(0).unique():
    # Compute symmetric component
    pv[(var, "sym")] = (pv[(var, "pos")] + pv[(var, "neg")]) / 2
    # Compute antisymmetric component
    pv[(var, "antisym")] = (pv[(var, "pos")] - pv[(var, "neg")]) / 2


In [2]:
## refactored:
def fold_data(df, fold_col, group_cols=None, columns_to_fold=None):
    # Error Check
    df = df.select_dtypes(include="number")  # Keep only numeric columns
    group_cols = group_cols if isinstance(group_cols, list) else [group_cols]  # Ensure group_cols is a list

    # Pivot
    df["sign"], df["abs"] = np.sign(df[fold_col]), np.abs(df[fold_col])
    pv = df.pivot_table(index=["abs"] + group_cols, columns="sign", values=columns_to_fold)

    # Compute
    for var in pv.columns.get_level_values(0).unique():
        pv[(var, "sym")] = (pv[(var, 1)] + pv[(var, -1)]) / 2  # Symmetrize
        pv[(var, "anti")] = (pv[(var, 1)] - pv[(var, -1)]) / 2  # Antisymmetrize
    pv.columns = ["_".join(map(str, col)) for col in pv.columns]  # Flatten column names

    return pv.reset_index()


# VScode tips

- `shift alt I` puts cursors at [end of all selected lines](https://stackoverflow.com/a/61177210/28244515)
- hit `home` after to get cursors at begining of all selected lines

In [ ]:
def load_calcResist(filename):
    df = rename_columns(pd.read_csv(filename, comment=";"))


    df["Vdc"] = df["myHF2LI.dc"].round(3)
    df["Vac"] = df["myHF2LI.ac"].round(3)

    #calc resistance
    Rext = 8.12e3 + 50; #print("external resistor + 50ohm lockin = ", Rext)
    df["Iac"] = df["myHF2LI.ac"] / Rext
    df["Idc"] = df["myHF2LI.dc"] / Rext
    df["R4"] = df.TX1 /df.Iac

    return df

- `alt uparrow` reorders selected line of code up or down
- `cntr alt uparrow` adds second cursor to line above
- `alt leftClick` adds second cursor to click location